### 1. Imports and Configuration

In [ ]:
import time
import json
import csv
import pickle
import random
from pathlib import Path

import numpy               as np
import torch
import torch.nn            as nn
import torch.nn.functional as F
from torch.utils.data      import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import datetime
from IPython.display        import clear_output
from sympy                  import nextprime, prime

# -----------------------------------------------------------------------
# Configuration -- mirrors notebook 00.
# -----------------------------------------------------------------------

VOCAB_SIZE      = 15
TOK_PAD         = 14
TOK_SEP         = 13
TOK_FIZZ        = 10
TOK_BUZZ        = 11
TOK_FZBZ        = 12
SEQ_LEN         = 128
EMBED_DIM       = 128
NUM_HEADS       = 8
NUM_LAYERS      = 8
FF_DIM          = 256
DROPOUT         = 0.1
TERNARY_THRESH  = 0.05
BATCH_SIZE      = 128
EPOCHS_FP32     = 200
EPOCHS_TERNARY  = 400
LR_FP32         = 3e-4
LR_TERNARY      = 1e-3
WEIGHT_DECAY    = 1e-4
EVAL_EVERY      = 10
SEED            = 42
TASK_NAMES      = ['fibonacci', 'fizzbuzz', 'parity', 'primes']

ROOT_DIR        = Path('..').resolve()
DATA_DIR        = ROOT_DIR / 'data'
CKPT_DIR        = ROOT_DIR / 'checkpoints'
METRICS_DIR     = ROOT_DIR / 'metrics'

for d in [CKPT_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------
# Device selection -- MPS is used when available on Apple Silicon to
# accelerate training via GPU compute. The purpose is fast iteration
# across multiple runs for comparison, not benchmarking training speed
# itself. The ternary model also runs on MPS since TernaryLinear.quantize()
# uses standard PyTorch ops that MPS supports. Falls back to CPU otherwise.
# -----------------------------------------------------------------------

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

# -----------------------------------------------------------------------
# Run versioning -- each training run writes to its own timestamped
# subdirectory so results never overwrite each other across experiments.
# -----------------------------------------------------------------------

RUN_NAME    = datetime.datetime.now().strftime('run_%Y%m%d_%H%M%S')
RUN_METRICS = METRICS_DIR / RUN_NAME
RUN_CKPT    = CKPT_DIR    / RUN_NAME
RUN_PLOTS   = RUN_METRICS / 'plots'

for d in [RUN_METRICS, RUN_CKPT, RUN_PLOTS]:
    d.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)

print(f'Config loaded  | Epochs FP32: {EPOCHS_FP32}  | Epochs Ternary: {EPOCHS_TERNARY}  |  Batch: {BATCH_SIZE}')
print(f'Ternary thresh : {TERNARY_THRESH}')
print(f'Device         : {DEVICE}')
print(f'Run name       : {RUN_NAME}')
print(f'Metrics dir    : {RUN_METRICS}')

### 2. Dataset Loading

In [2]:
# -----------------------------------------------------------------------
# Vocabulary helpers and task generators are redefined here so this
# notebook unpickles the dataset without importing notebook 01.
# -----------------------------------------------------------------------

def int_to_tokens(n):
    return [int(d) for d in str(n)]


def fizzbuzz_tokens(n):
    if n % 15 == 0: return [TOK_FZBZ]
    if n % 3  == 0: return [TOK_FIZZ]
    if n % 5  == 0: return [TOK_BUZZ]
    return int_to_tokens(n)


def gen_fibonacci(target_len):
    a, b = random.randint(0, 9), random.randint(0, 9)
    out  = []
    while len(out) < target_len:
        out.extend(int_to_tokens(a % 1000))
        out.append(TOK_SEP)
        a, b = b, (a + b) % 1000
    return out


def gen_fizzbuzz(target_len):
    n, out = random.randint(1, 200), []
    while len(out) < target_len:
        out.extend(fizzbuzz_tokens(n))
        out.append(TOK_SEP)
        n += 1
    return out


def gen_parity(target_len):
    out = []
    while len(out) < target_len:
        bits   = [random.randint(0, 1) for _ in range(random.randint(4, 14))]
        parity = sum(bits) % 2
        out.extend(bits)
        out.append(TOK_SEP)
        out.append(parity)
        out.append(TOK_SEP)
    return out


def gen_primes(target_len):
    p, out = prime(random.randint(1, 100)), []
    while len(out) < target_len:
        out.extend(int_to_tokens(p))
        out.append(TOK_SEP)
        p = nextprime(p)
    return out


TASK_GENERATORS = {
    'fibonacci' : gen_fibonacci,
    'fizzbuzz'  : gen_fizzbuzz,
    'parity'    : gen_parity,
    'primes'    : gen_primes,
}


class SyntheticTaskDataset(Dataset):
    '''
    Multi-task synthetic dataset -- see notebook 01 for full documentation.
    Redefined here so unpickling works without importing notebook 01.
    '''

    def __init__(self, n_samples, seq_len, seed=42):
        super().__init__()
        self.seq_len = seq_len
        per_task     = n_samples // len(TASK_NAMES)
        local_rng    = random.Random(seed)
        samples, labels = [], []

        for task_idx, name in enumerate(TASK_NAMES):
            gen = TASK_GENERATORS[name]
            for _ in range(per_task):
                tokens = gen(seq_len + 1)
                if len(tokens) < seq_len + 1:
                    tokens = tokens + [TOK_PAD] * (seq_len + 1 - len(tokens))
                samples.append(tokens[: seq_len + 1])
                labels.append(task_idx)

        combined = list(zip(samples, labels))
        local_rng.shuffle(combined)
        samples, labels  = zip(*combined)
        self.samples     = [torch.tensor(s, dtype=torch.long) for s in samples]
        self.labels      = list(labels)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        tokens = self.samples[idx]
        return tokens[:-1], tokens[1:], self.labels[idx]


# -----------------------------------------------------------------------
# Load the saved dataset and reconstruct train / val splits using the
# stored indices so the partition is identical to notebook 01.
# -----------------------------------------------------------------------

with open(DATA_DIR / 'synthetic_dataset.pkl', 'rb') as f:
    saved = pickle.load(f)

dataset       = saved['dataset']
train_indices = saved['train_indices']
val_indices   = saved['val_indices']

train_ds = Subset(dataset, train_indices)
val_ds   = Subset(dataset, val_indices)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Dataset loaded  : {len(dataset)} total samples')
print(f'Train           : {len(train_ds)}  |  Val: {len(val_ds)}')
print(f'Train batches   : {len(train_loader)}  |  Val batches: {len(val_loader)}')

Dataset loaded  : 10000 total samples
Train           : 9000  |  Val: 1000
Train batches   : 71  |  Val batches: 8


### 3. Model Definitions

In [3]:
# -----------------------------------------------------------------------
# Model definitions redefined here for self-contained execution.
# See notebook 02 for full documentation on each class.
# -----------------------------------------------------------------------

class TernaryLinear(nn.Linear):
    '''
    nn.Linear replacement with ternary weight quantization and STE.
    Quantization rule: +1 if w > thresh, 0 if |w| <= thresh, -1 if w < -thresh.
    STE trick: w + (q - w).detach() is q in the forward pass and w in the backward pass.
    '''

    def __init__(self, in_features, out_features, bias=True):
        super().__init__(in_features, out_features, bias)

    def quantize(self):
        w = self.weight
        q = torch.sign(w) * (w.abs() > TERNARY_THRESH).float()
        return w + (q - w).detach()

    def forward(self, x):
        return F.linear(x, self.quantize(), self.bias)


class TransformerBlock(nn.Module):
    '''
    Pre-norm causal transformer block. linear_cls selects nn.Linear
    (FP32) or TernaryLinear (ternary) for the feed-forward layers.
    '''

    def __init__(self, embed_dim, num_heads, ff_dim, dropout, linear_cls=nn.Linear):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff1   = linear_cls(embed_dim, ff_dim)
        self.ff2   = linear_cls(ff_dim, embed_dim)
        self.act   = nn.GELU()
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        h    = self.norm1(x)
        h, _ = self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)
        x    = x + self.drop(h)
        h    = self.norm2(x)
        h    = self.drop(self.act(self.ff1(h)))
        h    = self.ff2(h)
        x    = x + self.drop(h)
        return x


class TinyTransformer(nn.Module):
    '''
    Causal LM. Token embeddings and output head are always float32.
    linear_cls controls whether feed-forward layers are FP32 or ternary.
    '''

    def __init__(self, vocab_size, seq_len, embed_dim, num_heads,
                 num_layers, ff_dim, dropout, linear_cls=nn.Linear):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=TOK_PAD)
        self.pos_embed = nn.Embedding(seq_len,    embed_dim)
        self.layers    = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout, linear_cls)
            for _ in range(num_layers)
        ])
        self.norm      = nn.LayerNorm(embed_dim)
        self.output    = nn.Linear(embed_dim, vocab_size, bias=False)
        self.seq_len   = seq_len
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, x):
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0)
        h    = self.tok_embed(x) + self.pos_embed(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        for layer in self.layers:
            h = layer(h, attn_mask=mask)
        return self.output(self.norm(h))


# -----------------------------------------------------------------------
# Ternary metric helpers -- redefined from notebook 02.
# -----------------------------------------------------------------------

def get_ternary_stats(model):
    '''
    Fraction of ternary weights at -1 / 0 / +1 and mean latent variance.
    Returns a dict with keys: neg1_frac, zero_frac, pos1_frac, latent_var.
    '''
    neg1 = zero = pos1 = total = 0
    layer_vars = []
    for m in model.modules():
        if isinstance(m, TernaryLinear):
            w = m.weight.detach()
            q = torch.zeros_like(w)
            q[w >  TERNARY_THRESH] =  1.0
            q[w < -TERNARY_THRESH] = -1.0
            neg1  += (q == -1).sum().item()
            zero  += (q ==  0).sum().item()
            pos1  += (q ==  1).sum().item()
            total += q.numel()
            layer_vars.append(w.var().item())
    if total == 0:
        return {'neg1_frac': 0.0, 'zero_frac': 0.0, 'pos1_frac': 0.0, 'latent_var': 0.0}
    return {
        'neg1_frac' : neg1 / total,
        'zero_frac' : zero / total,
        'pos1_frac' : pos1 / total,
        'latent_var': float(sum(layer_vars) / len(layer_vars)),
    }


def get_ternary_snapshot(model):
    '''
    Capture quantized weight tensors for all TernaryLinear layers.
    Returns a dict mapping layer name to a detached quantized weight tensor.
    '''
    snap = {}
    for name, m in model.named_modules():
        if isinstance(m, TernaryLinear):
            w = m.weight.detach()
            q = torch.zeros_like(w)
            q[w >  TERNARY_THRESH] =  1.0
            q[w < -TERNARY_THRESH] = -1.0
            snap[name] = q.clone()
    return snap


def compute_weight_churn(model, prev_snapshot):
    '''
    Fraction of ternary weights that changed quantized value since prev_snapshot.
    Returns (churn_fraction, new_snapshot).
    '''
    curr    = get_ternary_snapshot(model)
    changed = total = 0
    for name, q in curr.items():
        if name in prev_snapshot:
            changed += (q != prev_snapshot[name]).sum().item()
            total   += q.numel()
    return (changed / total if total > 0 else 0.0), curr


# -----------------------------------------------------------------------
# Instantiate both models with the same seed for identical initial weights.
# -----------------------------------------------------------------------

MODEL_KWARGS = dict(
    vocab_size = VOCAB_SIZE,
    seq_len    = SEQ_LEN,
    embed_dim  = EMBED_DIM,
    num_heads  = NUM_HEADS,
    num_layers = NUM_LAYERS,
    ff_dim     = FF_DIM,
    dropout    = DROPOUT,
)

torch.manual_seed(SEED)
fp32_model    = TinyTransformer(**MODEL_KWARGS, linear_cls=nn.Linear).to(DEVICE)

torch.manual_seed(SEED)
ternary_model = TinyTransformer(**MODEL_KWARGS, linear_cls=TernaryLinear).to(DEVICE)

n_params = sum(p.numel() for p in fp32_model.parameters())
print(f'Both models instantiated  |  Parameters: {n_params:,}')

Both models instantiated  |  Parameters: 1,080,320


### 4. Training Utilities

In [4]:
# -----------------------------------------------------------------------
# train_epoch and eval_epoch are pure training functions with no display
# logic. All progress rendering is handled by render_progress() below.
# -----------------------------------------------------------------------

def train_epoch(model, loader, optimizer):
    '''
    Run one full training epoch.

    Computes cross-entropy loss over next-token predictions, ignoring
    PAD tokens. Gradient norms are recorded before clipping so they
    reflect the true gradient signal, not the clipped version.

    Parameters
    ----------
    model     : nn.Module
    loader    : DataLoader
    optimizer : torch.optim.Optimizer

    Returns
    -------
    tuple[float, float]
        (mean_loss, mean_grad_norm) averaged over all batches.
    '''
    model.train()
    total_loss = total_grad_norm = n_batches = 0.0

    for x, y, _ in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        logits = model(x)                                     # (B, T, V)

        # PAD positions are ignored -- parity samples are padded after
        # the answer token so the loss naturally falls only on the one
        # meaningful parity position, with no masking logic required.
        loss = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),
            y.reshape(-1),
            ignore_index = TOK_PAD,
        )
        
        loss.backward()

        # Record gradient norm BEFORE clipping
        grad_norm = sum(
            p.grad.norm().item() ** 2
            for p in model.parameters()
            if p.grad is not None
        ) ** 0.5

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss      += loss.item()
        total_grad_norm += grad_norm
        n_batches       += 1

    return total_loss / n_batches, total_grad_norm / n_batches


def eval_epoch(model, loader):
    '''
    Evaluate the model on the given loader without gradient computation.

    Parameters
    ----------
    model  : nn.Module
    loader : DataLoader

    Returns
    -------
    float
        Mean cross-entropy loss over all batches.
    '''
    model.eval()
    total_loss = n_batches = 0.0

    with torch.no_grad():
        for x, y, _ in loader:
            x, y   = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss   = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE),
                y.reshape(-1),
                ignore_index = TOK_PAD,
            )
            total_loss += loss.item()
            n_batches  += 1

    return total_loss / n_batches


def eval_per_task_accuracy(model, dataset, task_indices_map):
    '''
    Compute next-token prediction accuracy separately for each task.

    Accuracy is measured on non-PAD tokens only, using argmax over logits.

    Parameters
    ----------
    model            : nn.Module
    dataset          : Dataset
    task_indices_map : dict[str, list[int]]

    Returns
    -------
    dict[str, float]
        Accuracy per task in [0, 1].
    '''
    model.eval()
    results = {}

    with torch.no_grad():
        for task_name, indices in task_indices_map.items():
            task_loader = DataLoader(Subset(dataset, indices), batch_size=256, shuffle=False)
            correct = total = 0

            for x, y, _ in task_loader:
                x, y   = x.to(DEVICE), y.to(DEVICE)
                preds  = model(x).argmax(dim=-1)              # (B, T)
                mask   = (y != TOK_PAD)
                correct += (preds[mask] == y[mask]).sum().item()
                total   += mask.sum().item()

            results[task_name] = correct / total if total > 0 else 0.0

    return results


# -----------------------------------------------------------------------
# Pre-compute per-task val indices once -- avoids scanning the full
# dataset on every evaluation epoch.
# -----------------------------------------------------------------------

val_task_indices = {name: [] for name in TASK_NAMES}
for idx in val_indices:
    _, _, task_idx = dataset[idx]
    val_task_indices[TASK_NAMES[task_idx]].append(idx)

print('Per-task val split sizes:')
for name, indices in val_task_indices.items():
    print(f'  {name:<12}  {len(indices)} samples')

Per-task val split sizes:
  fibonacci     255 samples
  fizzbuzz      269 samples
  parity        224 samples
  primes        252 samples


### 5. Live Display

In [5]:
def render_progress(epoch, total_epochs, fp32_tr, fp32_val, tern_tr, tern_val,
                    epoch_time, total_elapsed, t_stats, churn, checkpoint_log):
    '''
    Overwrite the cell output with a live training summary.

    Called every epoch. clear_output(wait=True) holds the clear until
    the new content is ready so the cell does not flash blank.

    The checkpoint log is a list of strings accumulated over training.
    Only the last 20 lines are shown so the cell does not grow unbounded.

    Parameters
    ----------
    epoch          : int
    total_epochs   : int
    fp32_tr        : float   FP32 train loss.
    fp32_val       : float   FP32 val loss.
    tern_tr        : float   Ternary train loss.
    tern_val       : float   Ternary val loss.
    epoch_time     : float   Combined epoch time for both models (seconds).
    total_elapsed  : float   Wall time since training started (seconds).
    t_stats        : dict    Latest ternary weight stats (carries forward from last eval epoch).
    churn          : float   Latest weight churn value.
    checkpoint_log : list    Accumulated log lines from eval checkpoints.
    '''
    # ASCII progress bar
    bar_width = 42
    filled    = int(bar_width * epoch / total_epochs)
    bar       = '=' * filled + '>' + '.' * (bar_width - filled - 1)

    # ETA based on mean epoch time so far
    mean_time = total_elapsed / epoch
    eta_s     = mean_time * (total_epochs - epoch)
    eta_str   = f'{eta_s / 60:.1f} min' if eta_s >= 60 else f'{eta_s:.0f}s'

    clear_output(wait=True)

    # -- Progress bar and timing ----------------------------------------
    print(f'Epoch  {epoch:>4} / {total_epochs}  [{bar}]  {epoch / total_epochs * 100:.1f}%')
    print(f'Elapsed : {total_elapsed / 60:.1f} min  |  ETA : {eta_str}  |  Last epoch : {epoch_time:.1f}s')
    print()

    # -- Loss table -----------------------------------------------------
    print(f'  {"":20}  {"Train Loss":>12}  {"Val Loss":>10}')
    print(f'  {"-" * 46}')
    print(f'  {"FP32":<20}  {fp32_tr:>12.4f}  {fp32_val:>10.4f}')
    print(f'  {"Ternary":<20}  {tern_tr:>12.4f}  {tern_val:>10.4f}')
    print()

    # -- Ternary weight stats (carries forward so always visible) -------
    if t_stats is not None:
        print(f'  Weights   -1: {t_stats["neg1_frac"]:.3f}  '
              f'0: {t_stats["zero_frac"]:.3f}  '
              f'+1: {t_stats["pos1_frac"]:.3f}  '
              f'|  Churn: {churn:.3f}  '
              f'|  Latent var: {t_stats["latent_var"]:.5f}')
        print()

    # -- Checkpoint log (last 20 eval epochs) ---------------------------
    if checkpoint_log:
        print(f'  {"-" * 66}')
        for line in checkpoint_log[-20:]:
            print(line)


print('render_progress defined.')

render_progress defined.


### 6. Training Loop

In [6]:
# -----------------------------------------------------------------------
# Metrics storage -- all lists indexed by epoch (0-based).
# Ternary-specific keys are only populated on eval epochs.
# -----------------------------------------------------------------------

metrics = {
    'fp32': {
        'train_loss'     : [],
        'val_loss'       : [],
        'grad_norm'      : [],
        'epoch_time_s'   : [],
        'task_accuracy'  : {name: [] for name in TASK_NAMES},
        'task_acc_epochs': [],
    },
    'ternary': {
        'train_loss'     : [],
        'val_loss'       : [],
        'grad_norm'      : [],
        'epoch_time_s'   : [],
        'task_accuracy'  : {name: [] for name in TASK_NAMES},
        'task_acc_epochs': [],
        'neg1_frac'      : [],
        'zero_frac'      : [],
        'pos1_frac'      : [],
        'latent_var'     : [],
        'weight_churn'   : [],
        'ternary_epochs' : [],
    },
}

# AdamW optimizers 
fp32_optimizer    = torch.optim.AdamW(fp32_model.parameters(),    lr=LR_FP32,    weight_decay=WEIGHT_DECAY)
ternary_optimizer = torch.optim.AdamW(ternary_model.parameters(), lr=LR_TERNARY, weight_decay=WEIGHT_DECAY)

# -----------------------------------------------------------------------
# Cosine annealing with warm restarts -- decays LR smoothly from the
# initial value to near zero over T_0 epochs, then restarts. T_mult=2
# doubles the cycle length after each restart so early training gets
# aggressive LR changes and later training gets longer stable phases.
# -----------------------------------------------------------------------

fp32_scheduler    = CosineAnnealingWarmRestarts(fp32_optimizer,    T_0=50, T_mult=2, eta_min=1e-6)
ternary_scheduler = CosineAnnealingWarmRestarts(ternary_optimizer, T_0=50, T_mult=2, eta_min=1e-6)

# Initial ternary snapshot -- used to compute churn on the first eval epoch
prev_ternary_snapshot = get_ternary_snapshot(ternary_model)

# State carried between epochs so the live display always shows the
# most recent ternary stats even on non-eval epochs
last_t_stats   = None
last_churn     = None
checkpoint_log = []

total_start = time.perf_counter()

for epoch in range(1, EPOCHS_TERNARY + 1):

    # -- FP32 training epoch -- only runs for EPOCHS_FP32 epochs -------
    if epoch <= EPOCHS_FP32:
        t0                              = time.perf_counter()
        fp32_train_loss, fp32_grad_norm = train_epoch(fp32_model, train_loader, fp32_optimizer)
        fp32_val_loss                   = eval_epoch(fp32_model, val_loader)
        fp32_epoch_time                 = time.perf_counter() - t0
        fp32_scheduler.step()

        metrics['fp32']['train_loss'].append(fp32_train_loss)
        metrics['fp32']['val_loss'].append(fp32_val_loss)
        metrics['fp32']['grad_norm'].append(fp32_grad_norm)
        metrics['fp32']['epoch_time_s'].append(fp32_epoch_time)
    else:
        # FP32 training complete -- carry forward last values for display
        fp32_train_loss = metrics['fp32']['train_loss'][-1]
        fp32_val_loss   = metrics['fp32']['val_loss'][-1]
        fp32_epoch_time = 0.0

    # -- Ternary training epoch -- runs for all EPOCHS_TERNARY epochs --
    t0                              = time.perf_counter()
    tern_train_loss, tern_grad_norm = train_epoch(ternary_model, train_loader, ternary_optimizer)
    tern_val_loss                   = eval_epoch(ternary_model, val_loader)
    tern_epoch_time                 = time.perf_counter() - t0
    ternary_scheduler.step()

    metrics['ternary']['train_loss'].append(tern_train_loss)
    metrics['ternary']['val_loss'].append(tern_val_loss)
    metrics['ternary']['grad_norm'].append(tern_grad_norm)
    metrics['ternary']['epoch_time_s'].append(tern_epoch_time)

    total_elapsed = time.perf_counter() - total_start
    combined_time = fp32_epoch_time + tern_epoch_time

    # -- Periodic extended evaluation -----------------------------------
    if epoch % EVAL_EVERY == 0 or epoch == 1:

        # Only evaluate FP32 while it is still training
        if epoch <= EPOCHS_FP32:
            fp32_acc = eval_per_task_accuracy(fp32_model, dataset, val_task_indices)
            for name in TASK_NAMES:
                metrics['fp32']['task_accuracy'][name].append(fp32_acc[name])
            metrics['fp32']['task_acc_epochs'].append(epoch)

        ternary_acc = eval_per_task_accuracy(ternary_model, dataset, val_task_indices)
        for name in TASK_NAMES:
            metrics['ternary']['task_accuracy'][name].append(ternary_acc[name])
        metrics['ternary']['task_acc_epochs'].append(epoch)

        last_t_stats = get_ternary_stats(ternary_model)
        metrics['ternary']['neg1_frac'].append(last_t_stats['neg1_frac'])
        metrics['ternary']['zero_frac'].append(last_t_stats['zero_frac'])
        metrics['ternary']['pos1_frac'].append(last_t_stats['pos1_frac'])
        metrics['ternary']['latent_var'].append(last_t_stats['latent_var'])
        metrics['ternary']['ternary_epochs'].append(epoch)

        last_churn, prev_ternary_snapshot = compute_weight_churn(ternary_model, prev_ternary_snapshot)
        metrics['ternary']['weight_churn'].append(last_churn)

        fp32_acc_str  = '  '.join(f'{n[:3]}:{fp32_acc[n]:.2f}' for n in TASK_NAMES) if epoch <= EPOCHS_FP32 else 'complete'
        tern_acc_str  = '  '.join(f'{n[:3]}:{ternary_acc[n]:.2f}' for n in TASK_NAMES)
        fp32_str      = f'fp32 {fp32_train_loss:.4f}/{fp32_val_loss:.4f}' if epoch <= EPOCHS_FP32 else 'fp32 complete'
        checkpoint_log.append(
            f'  ep {epoch:>4}  '
            f'{fp32_str}  acc [{fp32_acc_str}]  '
            f'tern {tern_train_loss:.4f}/{tern_val_loss:.4f}  acc [{tern_acc_str}]'
        )

    render_progress(
        epoch, EPOCHS_TERNARY,
        fp32_train_loss, fp32_val_loss,
        tern_train_loss, tern_val_loss,
        combined_time, total_elapsed,
        last_t_stats, last_churn,
        checkpoint_log,
    )

print(f'Training complete  |  Total wall time: {total_elapsed / 60:.1f} min')

Epoch   400 / 400  [==========================================>]  100.0%
Elapsed : 442.4 min  |  ETA : 0s  |  Last epoch : 36.6s

                          Train Loss    Val Loss
  ----------------------------------------------
  FP32                        0.0553      0.0496
  Ternary                     0.0584      0.0500

  Weights   -1: 0.331  0: 0.341  +1: 0.328  |  Churn: 0.109  |  Latent var: 0.03822

  ------------------------------------------------------------------
  ep  210  fp32 complete  acc [complete]  tern 0.0618/0.0499  acc [fib:0.99  fiz:0.99  par:0.46  pri:0.99]
  ep  220  fp32 complete  acc [complete]  tern 0.0609/0.0500  acc [fib:0.99  fiz:0.99  par:0.45  pri:0.99]
  ep  230  fp32 complete  acc [complete]  tern 0.0597/0.0501  acc [fib:0.99  fiz:0.99  par:0.45  pri:0.99]
  ep  240  fp32 complete  acc [complete]  tern 0.0591/0.0500  acc [fib:0.99  fiz:0.99  par:0.46  pri:0.99]
  ep  250  fp32 complete  acc [complete]  tern 0.0582/0.0499  acc [fib:0.99  fiz:0.99  par:

### 7. Save Results

In [ ]:
# -----------------------------------------------------------------------
# Save run config -- records all hyperparameters so any run folder is
# fully self-documenting without needing to check the notebook source.
# -----------------------------------------------------------------------

run_config = {
    'run_name'          : RUN_NAME,
    'epochs_fp32'       : EPOCHS_FP32,
    'epochs_ternary'    : EPOCHS_TERNARY,
    'batch_size'        : BATCH_SIZE,
    'lr_fp32'           : LR_FP32,
    'lr_ternary'        : LR_TERNARY,
    'scheduler'         : 'CosineAnnealingWarmRestarts',
    'scheduler_T0'      : 50,
    'scheduler_T_mult'  : 2,
    'scheduler_eta_min' : 1e-6,
    'weight_decay'      : WEIGHT_DECAY,
    'ternary_thresh'    : TERNARY_THRESH,
    'embed_dim'         : EMBED_DIM,
    'num_heads'         : NUM_HEADS,
    'num_layers'        : NUM_LAYERS,
    'ff_dim'            : FF_DIM,
    'dropout'           : DROPOUT,
    'seq_len'           : SEQ_LEN,
    'vocab_size'        : VOCAB_SIZE,
    'n_samples'         : len(dataset),
    'train_samples'     : len(train_ds),
    'val_samples'       : len(val_ds),
    'device'            : str(DEVICE),
    'seed'              : SEED,
}

with open(RUN_METRICS / 'run_config.json', 'w') as f:
    json.dump(run_config, f, indent=2)

print(f'Config saved       : {RUN_METRICS / "run_config.json"}')

# -----------------------------------------------------------------------
# Save metrics as JSON (human-readable, diff-friendly for git) and model
# checkpoints as state dicts (portable, not tied to class definitions).
# -----------------------------------------------------------------------

metrics_path = RUN_METRICS / 'training_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'JSON saved         : {metrics_path}')

# -----------------------------------------------------------------------
# CSV export -- two files at different cadences for easy external inspection.
# Per-epoch CSV: one row per epoch, both model losses / grad norms / timing.
# Per-eval CSV:  one row per EVAL_EVERY checkpoint, task accuracy and
#                ternary weight distribution stats.
# -----------------------------------------------------------------------

epoch_csv_path = RUN_METRICS / 'metrics_per_epoch.csv'
with open(epoch_csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        'epoch',
        'fp32_train_loss', 'fp32_val_loss', 'fp32_grad_norm', 'fp32_epoch_time_s',
        'tern_train_loss', 'tern_val_loss', 'tern_grad_norm', 'tern_epoch_time_s',
    ])
    for i in range(len(metrics['fp32']['train_loss'])):
        writer.writerow([
            i + 1,
            metrics['fp32']['train_loss'][i],
            metrics['fp32']['val_loss'][i],
            metrics['fp32']['grad_norm'][i],
            metrics['fp32']['epoch_time_s'][i],
            metrics['ternary']['train_loss'][i],
            metrics['ternary']['val_loss'][i],
            metrics['ternary']['grad_norm'][i],
            metrics['ternary']['epoch_time_s'][i],
        ])

print(f'Epoch CSV saved    : {epoch_csv_path}')

eval_csv_path = RUN_METRICS / 'metrics_per_eval.csv'
with open(eval_csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        'epoch',
        'fp32_acc_fibonacci', 'fp32_acc_fizzbuzz', 'fp32_acc_parity', 'fp32_acc_primes',
        'tern_acc_fibonacci', 'tern_acc_fizzbuzz', 'tern_acc_parity', 'tern_acc_primes',
        'tern_neg1_frac', 'tern_zero_frac', 'tern_pos1_frac', 'tern_latent_var', 'tern_weight_churn',
    ])

    fp32_epoch_to_idx = {ep: idx for idx, ep in enumerate(metrics['fp32']['task_acc_epochs'])}

    for i, epoch in enumerate(metrics['ternary']['ternary_epochs']):
        fp32_idx = fp32_epoch_to_idx.get(epoch)
        fp32_vals = (
            [metrics['fp32']['task_accuracy'][name][fp32_idx] for name in TASK_NAMES]
            if fp32_idx is not None else [''] * len(TASK_NAMES)
        )
        writer.writerow([
            epoch,
            *fp32_vals,
            metrics['ternary']['task_accuracy']['fibonacci'][i],
            metrics['ternary']['task_accuracy']['fizzbuzz'][i],
            metrics['ternary']['task_accuracy']['parity'][i],
            metrics['ternary']['task_accuracy']['primes'][i],
            metrics['ternary']['neg1_frac'][i],
            metrics['ternary']['zero_frac'][i],
            metrics['ternary']['pos1_frac'][i],
            metrics['ternary']['latent_var'][i],
            metrics['ternary']['weight_churn'][i],
        ])

print(f'Eval CSV saved     : {eval_csv_path}')

fp32_ckpt_path    = RUN_CKPT / 'fp32_model.pt'
ternary_ckpt_path = RUN_CKPT / 'ternary_model.pt'

torch.save(fp32_model.state_dict(),    fp32_ckpt_path)
torch.save(ternary_model.state_dict(), ternary_ckpt_path)

print(f'FP32 checkpoint    : {fp32_ckpt_path}')
print(f'Ternary checkpoint : {ternary_ckpt_path}')

# -----------------------------------------------------------------------
# Final summary printed once after training is complete.
# -----------------------------------------------------------------------

print()
print(f'{"Metric":<30}  {"FP32":>10}  {"Ternary":>10}')
print('-' * 55)
print(f'{"Final train loss":<30}  {metrics["fp32"]["train_loss"][-1]:>10.4f}  {metrics["ternary"]["train_loss"][-1]:>10.4f}')
print(f'{"Final val loss":<30}  {metrics["fp32"]["val_loss"][-1]:>10.4f}  {metrics["ternary"]["val_loss"][-1]:>10.4f}')
print(f'{"Mean epoch time (s)":<30}  {sum(metrics["fp32"]["epoch_time_s"]) / EPOCHS_FP32:>10.3f}  {sum(metrics["ternary"]["epoch_time_s"]) / EPOCHS_TERNARY:>10.3f}')
print(f'{"Mean grad norm":<30}  {sum(metrics["fp32"]["grad_norm"]) / EPOCHS_FP32:>10.4f}  {sum(metrics["ternary"]["grad_norm"]) / EPOCHS_TERNARY:>10.4f}')
print()
print('Final per-task accuracy (val):')
print(f'{"Task":<15}  {"FP32":>10}  {"Ternary":>10}')
print('-' * 40)
for name in TASK_NAMES:
    print(f'{name:<15}  {metrics["fp32"]["task_accuracy"][name][-1]:>10.3f}  {metrics["ternary"]["task_accuracy"][name][-1]:>10.3f}')
print()
print('Final ternary weight distribution:')
print(f'  -1 : {metrics["ternary"]["neg1_frac"][-1]:.3f}')
print(f'   0 : {metrics["ternary"]["zero_frac"][-1]:.3f}')
print(f'  +1 : {metrics["ternary"]["pos1_frac"][-1]:.3f}')
print(f'  Final churn : {metrics["ternary"]["weight_churn"][-1]:.4f}')